# LU/LC Classification — Results: Visualizations and Tables

This notebook reads `results/tables/runlog.csv` and produces publication-ready tables and figures.

**Tables produced:**

| File | Contents |
|------|----------|
| `tbl_pred_pixel_counts.csv` | **Table 1** — Prediction map pixel distribution |
| `tbl_classwise_f1.csv` | **Table 2** — Class-wise F1 scores |
| `tbl_overall_metrics.csv` | **Table 3** — Overall accuracy metrics |

**Figures produced:**

| File | Contents |
|------|----------|
| `fig1_accuracy_trends.pdf` | **Figure 1** — Overall metric trends (by band configuration) |
| `fig2_classwise_f1_trends.pdf` | **Figure 2** — Class-wise F1 trends |
| `fig3_heatmap_by_algorithm.pdf` | **Figure 3** — Algorithm-wise F1 heatmap |
| `fig4_heatmap_by_bands.pdf` | **Figure 4** — Band configuration F1 heatmap |
| `fig5_feature_importance.pdf` | **Figure 5** — Normalized band importance |
| `fig6_delta_kappa_heatmap.pdf` | **Figure 6** — Marginal Kappa gain per added feature (ΔKappa) |
| `fig7_radar.pdf` | **Figure 7** — Algorithm class profiles on the final band configuration (radar) |

<br>

> `results/tables/runlog.csv` must exist before running this notebook.


In [ ]:
import sys
from pathlib import Path

# In Jupyter, __file__ is undefined; locate the project root from the current working directory
_candidates = [p for p in [Path().resolve()] + list(Path().resolve().parents)
               if (p / "config" / "hyperparameters.yaml").exists()]
if not _candidates:
    raise FileNotFoundError(
        "Project root not found. Launch Jupyter from inside the lulc-classification/ directory."
    )
PROJECT_ROOT = _candidates[0]

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.config import PATHS

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Global plot style — ensures consistent font across all figures
plt.rcParams.update({
    "font.family":    "sans-serif",
    "font.size":       10,
    "axes.titlesize":  11,
    "axes.labelsize":  10,
    "xtick.labelsize":  9,
    "ytick.labelsize":  9,
    "legend.fontsize":  9,
})

PATHS["figures"].mkdir(parents=True, exist_ok=True)
PATHS["tables"].mkdir(parents=True, exist_ok=True)

if not PATHS["runlog"].exists():
    raise FileNotFoundError(
        f"Run log not found: {PATHS['runlog']}\n"
        "Run 02_classify.ipynb for at least one band configuration first."
    )

df = pd.read_csv(PATHS["runlog"]).copy()
print(f"Run log loaded: {df.shape[0]} rows, {df.shape[1]} columns")
df.head()


In [ ]:
# Ordering and category definitions
# band_order: band labels that actually exist in the run log
# _all_bands = ["4 Band", "5 Band", "6 Band", "7 Band", "8 Band", "9 Band"]
_all_bands = ["4 Band", "5 Band", "6 Band", "9 Band"]
band_order = [b for b in _all_bands if b in df["Bands"].values]
alg_order  = ["AdaBoost", "CatBoost", "LightGBM", "XGBoost"]

df["Bands"]     = pd.Categorical(df["Bands"],     categories=band_order, ordered=True)
df["Algorithm"] = pd.Categorical(df["Algorithm"], categories=alg_order,  ordered=True)
df = df.sort_values(["Algorithm", "Bands"]).reset_index(drop=True)

# Extract class names dynamically from run log columns
# "Macro F1" and "Weighted F1" contain spaces, so they do not match the "_F1" suffix filter
classes = [c[:-3] for c in df.columns if c.endswith("_F1")]

# Keep only the bands listed in band_order; the categorical cast turned all others into NaN
df = df[df["Bands"].notna()].reset_index(drop=True)

# band_added: features introduced at each band configuration, derived dynamically from YAML
from src.config import load_config as _load_config
_label_to_bands = {v["label"]: v["band_names"]
                   for v in _load_config()["band_configs"].values()}

band_added = {}
for _i, _b in enumerate(band_order):
    _names = _label_to_bands.get(_b, [])
    if _i == 0:
        band_added[_b] = ", ".join(_names)
    else:
        _prev = set(_label_to_bands.get(band_order[_i - 1], []))
        _new  = [n for n in _names if n not in _prev]
        band_added[_b] = ("+ " + ", ".join(_new)) if _new else _b

print(f"Bands in run log: {band_order} ({len(band_order)} configurations)")
print(f"Total rows: {len(df)}")
print(f"Classes ({len(classes)}): {classes}")
print(f"band_added: {band_added}")
transitions = [(band_order[i], band_order[i+1], band_added[band_order[i+1]])
               for i in range(len(band_order) - 1)]
print(f"Transitions ({len(transitions)}): {transitions}")


## Table 1 — Prediction Map Pixel Distribution

Pixel counts and percentages per class for every prediction TIF in `results/predictions/`.


In [ ]:
# Read all prediction TIFs and combine class-wise pixel counts into a single table
# Wide format: rows = TIF names (algorithm + band config), columns = NoData + class names
# On every run, all current TIFs are scanned again so the table updates automatically

from src.metrics import CLASS_NAMES
from src.config import NODATA
from osgeo import gdal

pred_dir  = PATHS["predictions"]
pred_tifs = sorted(pred_dir.glob("*.tif")) if pred_dir.exists() else []
# Keep only TIFs that belong to the selected band configurations
_allowed_cfgs = {b.split()[0] + "b" for b in band_order}
pred_tifs = [p for p in pred_tifs if p.stem.split("_")[1] in _allowed_cfgs]

if not pred_tifs:
    print("No prediction TIF files found. Did you run 02_classify.ipynb?")
else:
    _row_data  = {}   # {tif_stem: {class_name: pixel_count}}
    _all_cols  = set()

    for tif_path in pred_tifs:
        row = tif_path.stem
        _row_data[row] = {}
        ds  = gdal.Open(str(tif_path))
        arr = ds.GetRasterBand(1).ReadAsArray().ravel()
        ds  = None
        vals, cnts = np.unique(arr, return_counts=True)
        for v, c in zip(vals, cnts):
            name = "NoData" if int(v) == NODATA else CLASS_NAMES.get(int(v), str(int(v)))
            _row_data[row][name] = int(c)
            _all_cols.add(name)

    # Column order: NoData first, then classes in alphabetical order
    _col_order = ["NoData"] + sorted(c for c in _all_cols if c != "NoData")

    # Wide-format DataFrame: rows = TIF names, columns = class names
    df_pred = pd.DataFrame.from_dict(_row_data, orient="index").reindex(columns=_col_order).fillna(0).astype(int)
    df_pred.index.name = "Algorithm"

    out = PATHS["tables"] / "tbl_pred_pixel_counts.csv"
    df_pred.to_csv(out)
    print(f"Saved: {out.name}  ({len(pred_tifs)} prediction TIFs)")
    display(df_pred)

## Table 2 — Class-wise F1 Scores


In [ ]:
f1_cols = [f"{c}_F1" for c in classes]

table_a = df[["Algorithm", "Bands"] + f1_cols].copy().round(2)
out_a = PATHS["tables"] / "tbl_classwise_f1.csv"
table_a.to_csv(out_a, index=False, float_format="%.2f")
print(f"Saved: {out_a}")
table_a

## Table 3 — Overall Accuracy Metrics


In [ ]:
overall_cols = ["Accuracy", "Weighted Precision", "Weighted Recall", "Weighted F1", "Kappa"]

table_b = df[["Algorithm", "Bands"] + overall_cols].copy().round(2)
out_b = PATHS["tables"] / "tbl_overall_metrics.csv"
table_b.to_csv(out_b, index=False, float_format="%.2f")
print(f"Saved: {out_b}")
table_b

## Figure 1 — Overall Metric Trends

Accuracy, Precision, Recall, F1, and Cohen's Kappa plotted against band configuration for each algorithm.


In [ ]:
colors = {
    "AdaBoost":  "#8ecae6",
    "CatBoost":  "#ff6b6b",
    "LightGBM":  "#8bd17c",
    "XGBoost":   "#f4a261",
}

metrics = [
    ("Cohen's Kappa", "Kappa",               (0.50, 1.00)),
    ("F1 Score",      "Weighted F1",         (0.50, 1.00)),
    ("Accuracy",      "Accuracy",            (0.50, 1.00)),
    ("Precision",     "Weighted Precision",  (0.50, 1.00)),
    ("Recall",        "Weighted Recall",     (0.50, 1.00)),
]

fig = plt.figure(figsize=(14, 8))
gs  = fig.add_gridspec(2, 6)
axes = [
    fig.add_subplot(gs[0, 0:3]),
    fig.add_subplot(gs[0, 3:6]),
    fig.add_subplot(gs[1, 0:2]),
    fig.add_subplot(gs[1, 2:4]),
    fig.add_subplot(gs[1, 4:6]),
]

for ax, (title, col, ylim) in zip(axes, metrics):
    for alg in alg_order:
        sub  = df[df["Algorithm"] == alg].sort_values("Bands")
        x    = sub["Bands"].astype(str).tolist()
        y    = sub[col].tolist()
        vals = ", ".join(f"{v:.2f}" for v in y)
        ax.plot(x, y, marker="o", linewidth=1.4, markersize=4,
                color=colors[alg], label=f"{alg} ({vals})")
    ax.set_title(title, fontsize=12)
    ax.set_ylim(*ylim)
    ax.grid(True, linestyle="--", linewidth=0.6, alpha=0.6)
    ax.legend(fontsize=10, loc="lower right", frameon=True)

fig.tight_layout()
out = PATHS["figures"] / "fig1_accuracy_trends.pdf"
fig.savefig(out, format="pdf", bbox_inches="tight")
plt.show()
print(f"Saved: {out}")

## Figure 2 — Class-wise F1 Trends (3×3 grid)

Per-class F1 score trends across band configurations, one subplot per class.


In [ ]:
n_cls  = len(classes)
n_cols = 3
n_rows = (n_cls + n_cols - 1) // n_cols  # ceil division

fig = plt.figure(figsize=(14, 4 * n_rows))
gs  = fig.add_gridspec(n_rows, n_cols)
axes = [fig.add_subplot(gs[r, c]) for r in range(n_rows) for c in range(n_cols)]

for i, cls in enumerate(classes):
    ax  = axes[i]
    col = f"{cls}_F1"
    for alg in alg_order:
        sub  = df[df["Algorithm"] == alg].sort_values("Bands")
        x    = sub["Bands"].astype(str).tolist()
        y    = sub[col].astype(float).tolist()
        vals = ", ".join(f"{v:.2f}" for v in y)
        ax.plot(x, y, marker="o", linewidth=1.4, markersize=4,
                color=colors[alg], label=f"{alg} ({vals})")
    ax.set_title(cls, fontsize=12)
    ax.set_ylim(0.0, 1.0)
    ax.grid(True, ls="--", alpha=0.5)
    ax.legend(fontsize=10, loc="lower right", frameon=True)

# Turn off any unused subplots
for j in range(n_cls, len(axes)):
    axes[j].axis("off")

plt.tight_layout()
out = PATHS["figures"] / "fig2_classwise_f1_trends.pdf"
fig.savefig(out, format="pdf", bbox_inches="tight")
plt.show()
print(f"Saved: {out}")

## Figure 3 — Algorithm-wise F1 Heatmap

Each subplot shows one algorithm. Rows = band configurations, columns = land-cover classes.
Diverging color scale centered at the dataset mean F1: red = above average, blue = below average.
Cell text shows the exact F1 score.


In [ ]:
import matplotlib.colors as mcolors

all_values = [v for cls in classes for v in df[f"{cls}_F1"].tolist()]
vmin, vmax = min(all_values), max(all_values)
vcenter    = (vmin + vmax) / 2
norm_f1    = mcolors.TwoSlopeNorm(vmin=vmin, vcenter=vcenter, vmax=vmax)

fig, axes = plt.subplots(4, 1, figsize=(6, max(3, len(band_order) * 1.5)))
plt.subplots_adjust(hspace=0.25)

for i, alg in enumerate(alg_order):
    sub = df[df["Algorithm"] == alg].sort_values("Bands")
    mat = sub[[f"{c}_F1" for c in classes]].to_numpy()
    actual_bands = sub["Bands"].astype(str).tolist()
    im = axes[i].imshow(mat, aspect="auto", cmap="plasma", norm=norm_f1)
    axes[i].set_title(alg)
    axes[i].set_yticks(range(len(actual_bands)))
    axes[i].set_yticklabels(actual_bands)
    if i == len(alg_order) - 1:
        axes[i].set_xticks(range(len(classes)))
        axes[i].set_xticklabels(classes, rotation=45, ha="right")
    else:
        axes[i].set_xticks([])
    for spine in axes[i].spines.values():
        spine.set_visible(False)
    for r in range(mat.shape[0]):
        for c in range(mat.shape[1]):
            val = mat[r, c]
            txt_color = "black" if val > 0.7 else "white"
            axes[i].text(
                c, r, f"{val:.2f}",
                ha="center", va="center",
                color=txt_color, fontsize=8
            )

cbar_ax = fig.add_axes([0.92, 0.15, 0.02, 0.7])
fig.colorbar(im, cax=cbar_ax).set_label("F1 Score")
plt.suptitle("Class-wise F1 Across Band Configurations", y=0.95)
plt.tight_layout(rect=[0, 0, 0.9, 0.96])

out = PATHS["figures"] / "fig3_heatmap_by_algorithm.pdf"
plt.savefig(out, dpi=300, bbox_inches="tight")
plt.show()
print(f"Saved: {out}")


## Figure 4 — Band Configuration F1 Heatmap

Each subplot shows one band configuration. Rows = algorithms, columns = land-cover classes.
Same color scale as Figure 3. Comparing rows within a subplot reveals which algorithm
best discriminates each class for a given band stack.


In [ ]:
# norm_f1 is defined in the Figure 3 cell
fig, axes = plt.subplots(len(band_order), 1,
                         figsize=(6, max(3, len(band_order) * 1.5)))
if len(band_order) == 1:
    axes = [axes]

for i, band in enumerate(band_order):
    sub = df[df["Bands"] == band].sort_values("Algorithm")
    mat = sub[[f"{c}_F1" for c in classes]].to_numpy()
    im  = axes[i].imshow(mat, aspect="auto", cmap="plasma", norm=norm_f1)
    axes[i].set_title(band)
    axes[i].set_yticks(range(len(alg_order)))
    axes[i].set_yticklabels(alg_order)
    axes[i].set_xticks(range(len(classes)))
    if i != len(band_order) - 1:
        axes[i].set_xticklabels([])
    else:
        axes[i].set_xticklabels(classes, rotation=45, ha="right")
    for spine in axes[i].spines.values():
        spine.set_visible(False)
    for r in range(mat.shape[0]):
        for c in range(mat.shape[1]):
            val = mat[r, c]
            txt_color = "black" if val > 0.7 else "white"
            axes[i].text(
                c, r, f"{val:.2f}",
                ha="center", va="center",
                color=txt_color, fontsize=8
            )

cbar_ax = fig.add_axes([0.92, 0.15, 0.02, 0.7])
fig.colorbar(im, cax=cbar_ax).set_label("F1 Score")
plt.suptitle("Effect of Band Count on Class Prediction", y=0.95)
plt.tight_layout(rect=[0, 0, 0.9, 0.96])

out = PATHS["figures"] / "fig4_heatmap_by_bands.pdf"
plt.savefig(out, dpi=300, bbox_inches="tight")
plt.show()
print(f"Saved: {out}")


## Figure 5 — Normalized Band Importance

Relative feature importance (split gain or permutation-based) normalized to sum to 1 per row. Values ≥ 5% are labelled.

> A dominant DEM or NDVI bar confirms the value of topographic/vegetation information beyond spectral bands.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick

from src.config import load_config as _load_config

# Collect FI columns dynamically from the run log
fi_cols = sorted(
    [c for c in df.columns if c.startswith("FI_")],
    key=lambda x: int(x.split("_")[1])
)
n_bands = len(fi_cols)

# Read band names from YAML; align with the number of FI columns
names_by_count = {
    v["num_bands"]: v["band_names"]
    for v in _load_config()["band_configs"].values()
}
if n_bands in names_by_count:
    band_labels = names_by_count[n_bands]
else:
    candidates = sorted((n, names) for n, names in names_by_count.items() if n >= n_bands)
    band_labels = (candidates[0][1][:n_bands] if candidates
                   else [f"Band{i + 1}" for i in range(n_bands)])

base_colors = [
    "#e63946", "#2a9d8f", "#3a86ff", "#8338ec",
    "#ffb703", "#52b788", "#495057", "#6c757d", "#adb5bd",
]
fi_colors = (base_colors * ((n_bands // len(base_colors)) + 1))[:n_bands]

alg_order_plot  = ["AdaBoost", "CatBoost", "LightGBM", "XGBoost"]
band_order_plot = ["4 band", "5 band", "6 band", "9 band"]

# Build normalised feature importance matrix — one row per (algorithm, band config) pair
plot_rows, plot_labels = [], []
for alg in alg_order_plot:
    sub = df[df["Algorithm"] == alg].copy()
    if sub.empty:
        continue
    sub["BandLabel"] = sub["Bands"].apply(lambda x: f"{int(str(x).split()[0])} band")
    sub["BandOrder"] = sub["BandLabel"].map({b: i for i, b in enumerate(band_order_plot)})
    sub = sub.dropna(subset=["BandOrder"]).sort_values("BandOrder")
    if sub.empty:
        continue
    raw = np.nan_to_num(sub[fi_cols].to_numpy(dtype=float), nan=0.0)
    for i, band_str in enumerate(sub["Bands"].astype(str)):
        valid_n = min(int(band_str.split()[0]), raw.shape[1])
        raw[i, valid_n:] = 0.0  # zero out bands absent from this configuration
    # Row-normalise so each bar sums to 100%
    s = raw.sum(axis=1, keepdims=True)
    mat = np.divide(raw, s, out=np.zeros_like(raw, dtype=float), where=s != 0)
    for i, (_, row) in enumerate(sub.iterrows()):
        plot_rows.append(mat[i])
        plot_labels.append(f"{alg} {int(str(row['Bands']).split()[0])} band")

plot_mat = np.vstack(plot_rows)

fig, ax = plt.subplots(figsize=(6.5, max(3, len(plot_labels) * 0.34)))

left  = np.zeros(plot_mat.shape[0])
y_pos = np.arange(plot_mat.shape[0])

for j in range(plot_mat.shape[1]):
    vals = plot_mat[:, j]
    ax.barh(y_pos, vals, left=left, height=0.80,
            color=fi_colors[j], edgecolor="none", label=band_labels[j])
    for k, v in enumerate(vals):
        if v >= 0.01:
            ax.text(left[k] + v / 2, y_pos[k], f"{v * 100:.0f}",
                    va="center", ha="center", fontsize=8,
                    fontweight="bold", color="white")
    left += vals

ax.set_yticks(y_pos)
ax.set_yticklabels(plot_labels, fontsize=7)
ax.invert_yaxis()
ax.set_xlim(0, 1)
ax.set_xticks([0, 0.25, 0.50, 0.75, 1.00])
ax.xaxis.set_major_formatter(mtick.PercentFormatter(xmax=1.0, decimals=0))
ax.set_xlabel("Normalized FI (%)")
ax.set_title("Normalized Band Importance Across Algorithms", fontsize=10)
ax.legend(loc="lower center", bbox_to_anchor=(0.5, -0.20),
           ncol=n_bands, frameon=False, fontsize=7)
for spine in ["top", "right"]:
    ax.spines[spine].set_visible(False)

plt.tight_layout()
out = PATHS["figures"] / "fig5_feature_importance.pdf"
plt.savefig(out, format="pdf", bbox_inches="tight")
plt.show()
print(f"Saved: {out}")

## Figure 6 — Marginal Kappa Gain per Added Feature (ΔKappa)

Change in Cohen's Kappa when each successive feature group is added to the band stack. Rows = algorithms, columns = added features. Cell text shows the exact ΔKappa value.


In [ ]:
# _step_pairs and _step_order are derived from transitions defined earlier in the notebook
_step_pairs = {(b1, b2): label for b1, b2, label in transitions}
_step_order = [label for _, _, label in transitions]

rows = []
for alg in alg_order:
    sub = df[df["Algorithm"] == alg].set_index("Bands")
    for (b1, b2), name in _step_pairs.items():
        delta = float(sub.loc[b2, "Kappa"]) - float(sub.loc[b1, "Kappa"])
        rows.append({"Algorithm": alg, "Feature": name, "ΔKappa": delta})

delta_kappa = (
    pd.DataFrame(rows)
    .pivot(index="Algorithm", columns="Feature", values="ΔKappa")
    .reindex(columns=_step_order)
    .reindex(alg_order)
)

fig, ax = plt.subplots(figsize=(max(4, len(_step_order) * 1.8), 3))
vmax = abs(delta_kappa.values).max()

cmap = plt.cm.plasma
norm = plt.Normalize(vmin=0, vmax=vmax)
im = ax.imshow(delta_kappa.values, cmap=cmap, norm=norm, aspect="auto")

for r in range(delta_kappa.shape[0]):
    for c in range(delta_kappa.shape[1]):
        val = delta_kappa.values[r, c]
        ax.text(c, r, f"{val:+.3f}", ha="center", va="center",
                fontsize=10, fontweight="normal",
                color="black" if abs(val) > vmax * 0.6 else "white")

ax.set_xticks(range(len(_step_order)))
ax.set_xticklabels(_step_order, rotation=0)
ax.set_yticks(range(len(alg_order)))
ax.set_yticklabels(alg_order)
ax.set_xlabel("Added Feature", fontsize=10)
ax.set_ylabel("")

for i in range(len(alg_order) + 1):
    ax.axhline(i - 0.5, color="white", lw=0.8)
for j in range(len(_step_order) + 1):
    ax.axvline(j - 0.5, color="white", lw=0.8)

cbar = fig.colorbar(im, ax=ax, shrink=0.85)
cbar.set_label("ΔKappa")
ax.set_title("Marginal Kappa Gain per Added Feature (ΔKappa)", fontsize=11)
for spine in ax.spines.values():
    spine.set_edgecolor("white")

plt.tight_layout()

fig_path = PATHS["figures"] / "fig6_delta_kappa_heatmap.pdf"
fig.savefig(fig_path, dpi=300, bbox_inches="tight")
plt.show()
print(f"Saved: {fig_path.name}")


## Figure 7 — Class-wise F1 on the Final Band Configuration (Radar)

Polar chart showing per-class F1 scores for all four algorithms on the full band stack.
Each axis = one land-cover class; each line traces an algorithm's class-level performance.


In [ ]:
_N11    = len(classes)
_angles = np.linspace(0, 2 * np.pi, _N11, endpoint=False).tolist()
_angles_closed = _angles + _angles[:1]

fig, ax = plt.subplots(figsize=(7, 7), subplot_kw=dict(polar=True))

for alg in alg_order:
    row  = df[(df["Algorithm"] == alg) & (df["Bands"] == band_order[-1])].iloc[0]
    vals = [float(row[f"{cls}_F1"]) for cls in classes]
    vals_closed = vals + vals[:1]
    ax.plot(_angles_closed, vals_closed, color=colors[alg], lw=1.5, label=alg, alpha=0.9)
    ax.fill(_angles_closed, vals_closed, color=colors[alg], alpha=0.01)

ax.set_ylim(0.3, 1.0)
ax.set_yticks([0.4, 0.6, 0.8, 1.0])
ax.set_yticklabels(["0.4", "0.6", "0.8", "1.0"], fontsize=7, color="#888")
ax.set_xticks(_angles)
ax.set_xticklabels(classes)
ax.tick_params(axis="x", pad=22)
ax.grid(color="gray", linestyle="--", linewidth=0.6, alpha=0.5)
ax.spines["polar"].set_visible(False)
ax.legend(loc="upper right", bbox_to_anchor=(1.32, 1.05),
          framealpha=0.9, title="Algorithm", frameon=False)
ax.set_title(f"Class-wise F1 profiles — {band_order[-1]}", pad=60)

fig_path = PATHS["figures"] / "fig7_radar.pdf"
fig.savefig(fig_path, dpi=300, bbox_inches="tight")
plt.show()
print(f"Saved: {fig_path.name}")
